<a href="https://colab.research.google.com/github/mondalankit665-gif/ankit-Mondal/blob/main/major%20project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
import joblib

# --- 1. DATA SIMULATION ENGINE ---
def prepare_engine_data(n=5000):
    rng = np.random.default_rng(42)
    data = pd.DataFrame({
        'recency': rng.integers(1, 180, n),
        'frequency': rng.integers(1, 50, n),
        'monetary': rng.uniform(100, 20000, n),
        'session_min': rng.uniform(5, 60, n),
        'tickets': rng.integers(0, 10, n),
        'clicks': rng.integers(0, 100, n)
    })
    # Target Logic
    churn_score = (data['recency'] * 0.5 + data['tickets'] * 20 - data['session_min'] * 5)
    data['churn'] = (churn_score > churn_score.median()).astype(int)
    data['clv'] = (data['monetary'] * 0.7 + data['frequency'] * 100).clip(0)
    return data

# --- 2. TRAIN MODELS ---
df_train = prepare_engine_data()
X = df_train.drop(['churn', 'clv'], axis=1)
churn_model = GradientBoostingClassifier().fit(X, df_train['churn'])
clv_model = RandomForestRegressor().fit(X, df_train['clv'])

joblib.dump(churn_model, 'churn_model.pkl')
joblib.dump(clv_model, 'clv_model.pkl')
print("Step 1: ML Engine ready. Models saved.")

In [12]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

st.set_page_config(page_title="Retention AI", layout="wide")

@st.cache_data
def get_dashboard_data():
    rng = np.random.default_rng(42)
    d = pd.DataFrame({
        'Customer ID': range(1001, 2001),
        'Recency': rng.integers(1, 180, 1000),
        'Frequency': rng.integers(1, 50, 1000),
        'Monetary': rng.uniform(100, 20000, 1000),
        'Churn_Prob': rng.uniform(0, 1, 1000),
        'CLV': rng.uniform(500, 25000, 1000)
    })
    d['Segment'] = pd.cut(d['Churn_Prob'], [0, 0.3, 0.7, 1.0], labels=['Stable', 'Warning', 'High Risk'])
    return d

data = get_dashboard_data()
st.title("📊 Retention & Revenue Impact Dashboard")

# KPIs
c1, c2, c3 = st.columns(3)
risky = data[data['Segment'] == 'High Risk']
c1.metric("High-Risk Churners", len(risky))
c2.metric("Revenue at Risk", f"${risky['Monetary'].sum():,.0f}")
c3.metric("Avg CLV", f"${data['CLV'].mean():,.0f}")

st.divider()
col1, col2 = st.columns(2)
with col1:
    st.subheader("Campaign Triggers")
    def get_action(row):
        if row['Segment'] == 'High Risk' and row['CLV'] > 15000: return "VIP: 25% Off + Support Call"
        if row['Segment'] == 'High Risk': return "Email: Re-engagement Sequence"
        return "Loyalty: Points Bonus"
    data['Action'] = data.apply(get_action, axis=1)
    st.dataframe(data[['Customer ID', 'Segment', 'CLV', 'Action']].head(15), use_container_width=True)
with col2:
    st.subheader("Customer RFM Map")
    fig = px.scatter(data, x='Recency', y='Frequency', size='Monetary', color='Segment')
    st.plotly_chart(fig, use_container_width=True)

Overwriting app.py


### 1. Advanced Data Simulation (RFM + CLV)
To predict churn effectively, we need features like Recency (days since last order), Frequency (total orders), and Monetary (total spend), alongside support and browsing history.

In [9]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

def generate_ecommerce_data(n=5000):
    rng = np.random.default_rng(42)
    data = pd.DataFrame({
        'customer_id': range(1001, 1001 + n),
        'recency': rng.integers(1, 180, n), # Days since last purchase
        'frequency': rng.integers(1, 50, n), # Total purchases
        'monetary': rng.uniform(100, 20000, n), # Total spend
        'avg_session_duration': rng.uniform(5, 60, n), # Minutes
        'support_tickets': rng.integers(0, 10, n),
        'email_clicks': rng.integers(0, 100, n),
        'last_30d_orders': rng.integers(0, 10, n)
    })

    # Target 1: Churn (Likely to not buy in next 30 days)
    # Logic: High recency + high support tickets + low session duration = Churn
    churn_prob = (data['recency'] * 0.4 + data['support_tickets'] * 15 - data['avg_session_duration'] * 2)
    data['churn'] = (churn_prob > churn_prob.median()).astype(int)

    # Target 2: CLV (Customer Lifetime Value)
    # Logic: Frequency * Monetary * session duration
    data['clv'] = (data['frequency'] * 0.5 + data['monetary'] * 0.8 + rng.normal(0, 500, n)).clip(0)

    return data

df = generate_ecommerce_data()
display(df.head())

,customer_id,recency,frequency,monetary,avg_session_duration,support_tickets,email_clicks,last_30d_orders,churn,clv
0,1001,16,10,4441.168773,44.640061,2,84,9,0,3850.003569
1,1002,139,2,13292.261241,44.118077,3,45,7,0,10261.404799
2,1003,118,31,15360.544254,16.148227,8,69,2,1,12751.231706
3,1004,79,34,3416.059246,7.016047,1,96,9,0,1934.763292
4,1005,78,38,972.779840,21.708974,1,4,9,0,748.018900


### 2. Training the Prediction Engine
We need two models: one for Churn (Classification) and one for CLV (Regression).

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop(['customer_id', 'churn', 'clv'], axis=1)
y_churn = df['churn']
y_clv = df['clv']

# Churn Model
churn_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
churn_model.fit(X, y_churn)

# CLV Model
clv_model = RandomForestRegressor(n_estimators=100, random_state=42)
clv_model.fit(X, y_clv)

print(f"Models Trained. Churn Features Importance: {churn_model.feature_importances_}")

Models Trained. Churn Features Importance: [1.32335561e-01 1.22071166e-04 6.79981927e-05 2.90477473e-01
 5.76868799e-01 7.32562071e-05 5.48423359e-05]


### 3. Automated Retention Campaigns
We categorize users and assign automated actions based on their risk level and value.

In [11]:
def get_retention_strategy(row):
    if row['churn'] == 1 and row['clv'] > df['clv'].quantile(0.8):
        return "VIP Retention: High-value at risk. Trigger 20% discount + Phone call."
    elif row['churn'] == 1:
        return "Standard Re-engagement: Trigger personalized email recommendations."
    else:
        return "Loyalty Building: Offer early access to new collections."

df['strategy'] = df.apply(get_retention_strategy, axis=1)
display(df[['customer_id', 'churn', 'clv', 'strategy']].head(10))

,customer_id,churn,clv,strategy
0,1001,0,3850.003569,Loyalty Building: Offer early access to new co...
1,1002,0,10261.404799,Loyalty Building: Offer early access to new co...
2,1003,1,12751.231706,VIP Retention: High-value at risk. Trigger 20%...
3,1004,0,1934.763292,Loyalty Building: Offer early access to new co...
4,1005,0,748.018900,Loyalty Building: Offer early access to new co...
5,1006,1,7084.612606,Standard Re-engagement: Trigger personalized e...
6,1007,1,12655.196998,Standard Re-engagement: Trigger personalized e...
7,1008,1,9945.334984,Standard Re-engagement: Trigger personalized e...
8,1009,1,10288.574958,Standard Re-engagement: Trigger personalized e...
9,1010,0,6585.838673,Loyalty Building: Offer early access to new co...


In [ ]:
# 1. Install Dependencies
!pip install -q streamlit plotly joblib
!npm install -g localtunnel -q

# 2. Launch Streamlit
import subprocess
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])

# 3. Create Tunnel Link
print("\n1. Copy this IP address:")
!curl ipv4.icanhazip.com
print("\n2. Click the link below, paste the IP, and hit 'Click to Submit':")
!npx localtunnel --port 8501

In [ ]:
# 1. Install localtunnel to expose the port
!npm install -g localtunnel -q

# 2. Run streamlit in the background
import subprocess
subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'])

# 3. Create a tunnel to the streamlit port
# Click the link generated below to view your dashboard.
!npx localtunnel --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 4s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠙⠙⠹⠸⠼⠴⠦⠧your url is: https://weak-sheep-deny.loca.lt
